In [1]:
!pip install pycountry

# Then import the required libraries
import pandas as pd
import streamlit as st
import plotly.express as px
import pycountry

In [2]:
vdem_df = pd.read_csv("data/raw/vdem_core.csv")
print(vdem_df.columns[:20])
print(vdem_df.shape)

Index(['country_name', 'country_text_id', 'country_id', 'year',
       'historical_date', 'project', 'historical', 'histname', 'codingstart',
       'codingend', 'codingstart_contemp', 'codingend_contemp',
       'codingstart_hist', 'codingend_hist', 'gapstart1', 'gapstart2',
       'gapstart3', 'gapend1', 'gapend2', 'gapend3'],
      dtype='object')
(28092, 1908)


In [3]:
keep = [
    "country_name",
    "country_text_id",
    "year",
    "v2x_libdem",
    "v2x_polyarchy",
    "v2x_partipdem",
    "v2x_egaldem",
    "v2x_delibdem"
]

panel = vdem_df[keep].copy()
panel = panel.dropna(subset=["country_name", "year"])
panel.to_csv("data/processed/vdem_panel.csv", index=False)

In [5]:
def to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except:
        return None

vdem_df["country_code"] = vdem_df["country_name"].apply(to_iso3)

In [6]:
#ERT Data Transformation
ert = pd.read_csv("data/raw/ert.csv")

# Inspect first
print("Columns:", ert.columns.tolist())
print("Shape:", ert.shape)
print(ert.head())

# Clean and standardize names (adjust based on your CSV)
ert.columns = ert.columns.str.lower().str.strip()

# Create country-year panel (expand episodes)
years = range(1960, 2026)  # Adjust range as needed
countries = ert["country_text_id"].unique()

panel = pd.DataFrame(index=pd.MultiIndex.from_product([countries, years], names=["country_text_id", "year"]))

# Merge episode info
for col in ert.columns:
    panel[col] = ert.set_index(["country_text_id"])[col].reindex(panel.index.get_level_values(0), method="ffill")

# Create binary flags for episodes (if not already present)
panel["dem_ep"] = 0
panel["aut_ep"] = 0

# Fill episode years based on start/end (adjust column names to match your CSV)
for idx, row in ert.iterrows():
    c = row["country_text_id"]
    dem_start = row.get("dem_ep_start_year", row.get("dem_start_year"))
    dem_end = row.get("dem_ep_end_year", row.get("dem_end_year"))
    aut_start = row.get("aut_ep_start_year", row.get("aut_start_year"))
    aut_end = row.get("aut_ep_end_year", row.get("aut_end_year"))
    
    if pd.notna(dem_start) and pd.notna(dem_end):
        mask = (panel.index.get_level_values(0) == c) & \
               (panel.index.get_level_values(1) >= dem_start) & \
               (panel.index.get_level_values(1) <= dem_end)
        panel.loc[mask, "dem_ep"] = 1
    
    if pd.notna(aut_start) and pd.notna(aut_end):
        mask = (panel.index.get_level_values(0) == c) & \
               (panel.index.get_level_values(1) >= aut_start) & \
               (panel.index.get_level_values(1) <= aut_end)
        panel.loc[mask, "aut_ep"] = 1

panel = panel.reset_index()

# Save processed panel
panel.to_csv("data/processed/ert_panel.csv", index=False)
print("ERT panel saved to data/processed/ert_panel.csv")
print(panel.groupby("year")["aut_ep"].sum().head())

Columns: ['Unnamed: 0', 'country_id', 'country_text_id', 'country_name', 'year', 'v2x_regime', 'v2x_polyarchy', 'v2x_polyarchy_codelow', 'v2x_polyarchy_codehigh', 'reg_start_year', 'reg_end_year', 'reg_id', 'reg_type', 'reg_trans', 'dem_founding_elec', 'aut_founding_elec', 'row_regch_event', 'row_regch_censored', 'dem_ep', 'dem_ep_id', 'dem_ep_start_year', 'dem_ep_end_year', 'dem_pre_ep_year', 'dem_ep_termination', 'dem_ep_prch', 'dem_ep_ptr', 'dem_ep_subdep', 'dem_ep_outcome', 'dem_ep_outcome_agg', 'dem_ep_censored', 'aut_ep', 'aut_ep_id', 'aut_ep_start_year', 'aut_ep_end_year', 'aut_pre_ep_year', 'aut_ep_termination', 'aut_ep_prch', 'aut_ep_pbr', 'aut_ep_subreg', 'aut_ep_outcome', 'aut_ep_outcome_agg', 'aut_ep_censored']
Shape: (19857, 42)
   Unnamed: 0  country_id country_text_id country_name  year  v2x_regime  \
0           1           3             MEX       Mexico  1900         0.0   
1           2           3             MEX       Mexico  1901         0.0   
2           3       

ValueError: cannot reindex on an axis with duplicate labels